In [1]:
!pip install transformers

In [2]:
!pip install -U bitsandbytes accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.3 MB/s eta 0:00:00


# Split dataset into 70 20 10 + balanced

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# reproducibility
np.random.seed(42)

# load data
df = pd.concat([
    pd.read_csv("train_clean.csv"),
    pd.read_csv("test_clean.csv"),
    pd.read_csv("validation_clean.csv")
], ignore_index=True)

# build document-level label
doc_labels = df.groupby("document_id")["entity_type"].unique().reset_index()

# convert label list → string (for stratification)
doc_labels["label"] = doc_labels["entity_type"].apply(lambda x: "_".join(sorted(x)))

# split doc_ids
train_ids, temp_ids = train_test_split(
    doc_labels["document_id"],
    test_size=0.3,
    stratify=doc_labels["label"],
    random_state=42
)

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=1/3,  # 0.1 total
    stratify=doc_labels.set_index("document_id").loc[temp_ids]["label"],
    random_state=42
)

# build final datasets
train_df = df[df["document_id"].isin(train_ids)]
val_df = df[df["document_id"].isin(val_ids)]
test_df = df[df["document_id"].isin(test_ids)]

# save
train_df.to_csv("train_70_balanced.csv", index=False)
val_df.to_csv("validation_20_balanced.csv", index=False)
test_df.to_csv("test_10_balanced.csv", index=False)

# check distribution
print("\nDistribution:")
print("Train:\n", train_df["entity_type"].value_counts(normalize=True))
print("\nValidation:\n", val_df["entity_type"].value_counts(normalize=True))
print("\nTest:\n", test_df["entity_type"].value_counts(normalize=True))


Distribution:
Train:
 entity_type
Chemical    0.541866
Disease     0.458134
Name: proportion, dtype: float64

Validation:
 entity_type
Chemical    0.550802
Disease     0.449198
Name: proportion, dtype: float64

Test:
 entity_type
Chemical    0.5299
Disease     0.4701
Name: proportion, dtype: float64


In [4]:
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 20243
Validation: 6171
Test: 3010


#library importation

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

🟦 🔹 1. LOAD FINAL SPLIT DATA

In [6]:
import pandas as pd

train = pd.read_csv("train_70_balanced.csv")
val = pd.read_csv("validation_20_balanced.csv")
test = pd.read_csv("test_10_balanced.csv")

datasets = {
    "Train": train,
    "Validation": val,
    "Test": test
}

🟦 🔹 2. EDA (like you already did but clean version)

In [7]:
for name, df in datasets.items():
    print(f"\n===== {name} HEAD =====")
    display(df.head())

    print(f"\n===== {name} INFO =====")
    print(df.info())

    print(f"\n===== {name} MISSING =====")
    print(df.isnull().sum())

    print(f"\n===== {name} CLASS DISTRIBUTION =====")
    print(df["entity_type"].value_counts())


===== Train HEAD =====


,document_id,passage_type,text,entity_text,entity_type,start,end
0,227508,title,Naloxone reverses the antihypertensive effect ...,Naloxone,Chemical,0,8
1,227508,title,Naloxone reverses the antihypertensive effect ...,clonidine,Chemical,49,58
2,227508,abstract,"In unanesthetized, spontaneously hypertensive ...",hypertensive,Disease,93,105
3,227508,abstract,"In unanesthetized, spontaneously hypertensive ...",clonidine,Chemical,181,190
4,227508,abstract,"In unanesthetized, spontaneously hypertensive ...",nalozone,Chemical,244,252



===== Train INFO =====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20243 entries, 0 to 20242
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   document_id   20243 non-null  int64 
 1   passage_type  20243 non-null  object
 2   text          20243 non-null  object
 3   entity_text   20243 non-null  object
 4   entity_type   20243 non-null  object
 5   start         20243 non-null  int64 
 6   end           20243 non-null  int64 
dtypes: int64(3), object(4)
memory usage: 1.1+ MB
None

===== Train MISSING =====
document_id     0
passage_type    0
text            0
entity_text     0
entity_type     0
start           0
end             0
dtype: int64

===== Train CLASS DISTRIBUTION =====
entity_type
Chemical    10969
Disease      9274
Name: count, dtype: int64

===== Validation HEAD =====


,document_id,passage_type,text,entity_text,entity_type,start,end
0,354896,title,Lidocaine-induced cardiac asystole.,Lidocaine,Chemical,0,9
1,354896,title,Lidocaine-induced cardiac asystole.,cardiac asystole,Disease,18,34
2,354896,abstract,Intravenous administration of a single 50-mg b...,lidocaine,Chemical,90,99
3,354896,abstract,Intravenous administration of a single 50-mg b...,depression,Disease,142,152
4,354896,abstract,Intravenous administration of a single 50-mg b...,bradyarrhythmias,Disease,331,347



===== Validation INFO =====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6171 entries, 0 to 6170
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   document_id   6171 non-null   int64 
 1   passage_type  6171 non-null   object
 2   text          6171 non-null   object
 3   entity_text   6164 non-null   object
 4   entity_type   6171 non-null   object
 5   start         6171 non-null   int64 
 6   end           6171 non-null   int64 
dtypes: int64(3), object(4)
memory usage: 337.6+ KB
None

===== Validation MISSING =====
document_id     0
passage_type    0
text            0
entity_text     7
entity_type     0
start           0
end             0
dtype: int64

===== Validation CLASS DISTRIBUTION =====
entity_type
Chemical    3399
Disease     2772
Name: count, dtype: int64

===== Test HEAD =====


,document_id,passage_type,text,entity_text,entity_type,start,end
0,2515254,title,Source of pain and primitive dysfunction in mi...,pain,Disease,10,14
1,2515254,title,Source of pain and primitive dysfunction in mi...,migraine,Disease,44,52
2,2515254,abstract,Twenty common migraine patients received a one...,migraine,Disease,87,95
3,2515254,abstract,Twenty common migraine patients received a one...,nitroglycerin,Chemical,156,169
4,2515254,abstract,Twenty common migraine patients received a one...,migraine,Disease,255,263



===== Test INFO =====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3010 entries, 0 to 3009
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   document_id   3010 non-null   int64 
 1   passage_type  3010 non-null   object
 2   text          3010 non-null   object
 3   entity_text   3010 non-null   object
 4   entity_type   3010 non-null   object
 5   start         3010 non-null   int64 
 6   end           3010 non-null   int64 
dtypes: int64(3), object(4)
memory usage: 164.7+ KB
None

===== Test MISSING =====
document_id     0
passage_type    0
text            0
entity_text     0
entity_type     0
start           0
end             0
dtype: int64

===== Test CLASS DISTRIBUTION =====
entity_type
Chemical    1595
Disease     1415
Name: count, dtype: int64


we found missing values in entity text in validation dataset


In [8]:
print("Before:", len(val))
val = val.dropna(subset=["entity_text"])
print("After:", len(val))

Before: 6171
After: 6164


🟦 🔹 3. GROUP ENTITIES (VERY IMPORTANT)

In [9]:
def group_entities_fixed(df):
    grouped = df.groupby("text")

    data = []
    for text, group in grouped:
        entities = []

        for _, row in group.iterrows():
            entity_text = row["entity_text"]

            # recompute start using real text
            start = text.find(entity_text)

            if start == -1:
                continue  # skip broken cases

            end = start + len(entity_text)

            entities.append({
                "start": start,
                "end": end,
                "label": row["entity_type"]
            })

        data.append({
            "text": text,
            "entities": entities
        })

    return data


train_grouped = group_entities_fixed(train)
val_grouped = group_entities_fixed(val)
test_grouped = group_entities_fixed(test)

display


In [10]:
def show_examples(grouped_data, num_examples=2):
    for i in range(num_examples):
        example = grouped_data[i]

        print(f"\n========== Example {i+1} ==========")
        print("TEXT:\n")
        print(example["text"])

        print("\nENTITIES:\n")
        for ent in example["entities"]:
            print(f"  - Label: {ent['label']}")
            print(f"    Start: {ent['start']}, End: {ent['end']}")
            print(f"    Text: '{example['text'][ent['start']:ent['end']]}'")
            print()

# Show examples
show_examples(train_grouped, 2)


========== Example 1 ==========
TEXT:

-Tacrolimus (FK 506) is a powerful, widely used immunosuppressant. The clinical utility of FK 506 is complicated by substantial hypertension and nephrotoxicity. To clarify the mechanisms of FK 506-induced hypertension, we studied the chronic effects of FK 506 on the synthesis of endothelin-1 (ET-1), the expression of mRNA of ET-1 and endothelin-converting enzyme-1 (ECE-1), the endothelial nitric oxide synthase (eNOS) activity, and the expression of mRNA of eNOS and C-type natriuretic peptide (CNP) in rat blood vessels. In addition, the effect of the specific endothelin type A receptor antagonist FR 139317 on FK 506-induced hypertension in rats was studied. FK 506, 5 mg. kg-1. d-1 given for 4 weeks, elevated blood pressure from 102+/-13 to 152+/-15 mm Hg and increased the synthesis of ET-1 and the levels of ET-1 mRNA in the mesenteric artery (240% and 230%, respectively). Little change was observed in the expression of ECE-1 mRNA and CNP mRNA. FK 

🟦 🔹 4. CONVERT TO BIO

In [11]:
import re

def text_to_bio(text, entities):
    tokens = []
    labels = []

    # 🔥 better tokenization (separates punctuation)
    for match in re.finditer(r'\w+|[^\w\s]', text):
        token = match.group()
        start = match.start()
        end = match.end()

        label = "O"

        for ent in entities:
            ent_start = ent["start"]
            ent_end = ent["end"]
            ent_label = ent["label"]

            # overlap
            if not (end <= ent_start or start >= ent_end):

                # ONLY assign label if token is NOT punctuation
                if re.match(r'\w+', token):
                    if start == ent_start:
                        label = "B-" + ent_label
                    else:
                        label = "I-" + ent_label

        tokens.append(token)
        labels.append(label)

    return tokens, labels

🟦 🔹 5. APPLY BIO

In [12]:
def convert_dataset(grouped_data):
    dataset = []

    for item in grouped_data:
        tokens, labels = text_to_bio(item["text"], item["entities"])

        dataset.append({
            "tokens": tokens,
            "labels": labels
        })

    return dataset

train_bio = convert_dataset(train_grouped)
val_bio = convert_dataset(val_grouped)
test_bio = convert_dataset(test_grouped)

🟦 🔹 6. CHECK RESULT

In [13]:
def show_bio_examples(dataset, num_examples=2):
    for i in range(num_examples):
        example = dataset[i]

        print(f"\n========== BIO Example {i+1} ==========\n")

        tokens = example["tokens"]
        labels = example["labels"]

        for token, label in zip(tokens, labels):
            print(f"{token:<20} -> {label}")

In [14]:
show_bio_examples(train_bio, 2)


========== BIO Example 1 ==========

-                    -> O
Tacrolimus           -> B-Chemical
(                    -> O
FK                   -> B-Chemical
506                  -> I-Chemical
)                    -> O
is                   -> O
a                    -> O
powerful             -> O
,                    -> O
widely               -> O
used                 -> O
immunosuppressant    -> O
.                    -> O
The                  -> O
clinical             -> O
utility              -> O
of                   -> O
FK                   -> O
506                  -> O
is                   -> O
complicated          -> O
by                   -> O
substantial          -> O
hypertension         -> B-Disease
and                  -> O
nephrotoxicity       -> B-Disease
.                    -> O
To                   -> O
clarify              -> O
the                  -> O
mechanisms           -> O
of                   -> O
FK                   -> O
506                  -> O
-        

In [15]:
# Installation de bitsandbytes pour la quantification 4-bit
!pip install bitsandbytes accelerate -q

In [16]:
!pip uninstall bitsandbytes -y
!pip install bitsandbytes>=0.46.1 accelerate -q

Found existing installation: bitsandbytes 0.49.2
Uninstalling bitsandbytes-0.49.2:
  Successfully uninstalled bitsandbytes-0.49.2


  7. tokenization


In [17]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import re
import json
from tqdm import tqdm
import numpy as np

model_name = "mistralai/Mistral-7B-Instruct-v0.3"

# Configuration pour charge en 4 bits
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Chargement du tokenizer Mistral...")
tokenizer_mistral = AutoTokenizer.from_pretrained(model_name)
tokenizer_mistral.pad_token = tokenizer_mistral.eos_token

print("Chargement du modèle Mistral-7B (4-bit) ...")
model_mistral = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model_mistral.eval()
print("Modèle Mistral-7B prêt.")

Chargement du tokenizer Mistral...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

... (3 outputs left out)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Modèle Mistral-7B prêt.


In [18]:
def extract_entities_mistral(text):
    prompt = f"""<s>[INST] You are a biomedical NER expert. Extract all chemicals and diseases from the text.
Return ONLY a JSON array. Example: [{{"text": "Lisinopril", "label": "Chemical"}}, {{"text": "diabetes", "label": "Disease"}}]

Text: {text}

Output: [/INST]"""

    inputs = tokenizer_mistral(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model_mistral.device)
    with torch.no_grad():
        outputs = model_mistral.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer_mistral.eos_token_id
        )
    response = tokenizer_mistral.decode(outputs[0], skip_special_tokens=True)

    try:
        json_part = response.split("[/INST]")[-1].strip()
        json_part = re.sub(r'^```json\s*|\s*```$', '', json_part)

        # Trouver le premier [ et le dernier ]
        start_idx = json_part.find('[')
        end_idx = json_part.rfind(']')
        if start_idx != -1 and end_idx != -1:
            json_part = json_part[start_idx:end_idx+1]

        entities = json.loads(json_part)
        if not isinstance(entities, list):
            entities = []
        entities = [e for e in entities if isinstance(e, dict) and "text" in e and "label" in e]
    except Exception as e:
        # Fallback: extraire avec regex
        pattern = r'"text"\s*:\s*"([^"]+)"\s*,\s*"label"\s*:\s*"([^"]+)"'
        matches = re.findall(pattern, json_part)
        entities = [{"text": m[0], "label": m[1]} for m in matches]
        if not entities:
            print(f"Erreur: {str(e)[:50]}")
            entities = []

    return entities

In [19]:
def text_to_bio(text, entities):
    tokens = []
    labels = []
    for match in re.finditer(r'\w+|[^\w\s]', text):
        token = match.group()
        start = match.start()
        end = match.end()
        label = "O"
        for ent in entities:
            ent_start = ent["start"]
            ent_end = ent["end"]
            ent_label = ent["label"]
            if not (end <= ent_start or start >= ent_end):
                if re.match(r'\w+', token):
                    if start == ent_start:
                        label = "B-" + ent_label
                    else:
                        label = "I-" + ent_label
        tokens.append(token)
        labels.append(label)
    return tokens, labels

In [20]:
!pip install seqeval -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [21]:
# ============================================
# INFÉRENCE + MÉTRIQUES MISTRAL-7B SUR BC5CDR
# ============================================

from seqeval.metrics import classification_report, f1_score, precision_score, recall_score, accuracy_score
from collections import Counter
from tqdm import tqdm
import re

# Définir les labels valides pour BC5CDR
VALID_LABELS = {"B-Chemical", "I-Chemical", "B-Disease", "I-Disease"}

# Fonction pour nettoyer les séquences
def clean_sequence(labels):
    """Remplace tout label invalide par O"""
    return [l if l in VALID_LABELS else "O" for l in labels]

# ============================================
# INFÉRENCE SUR LE TEST SET
# ============================================

true_labels = []
pred_labels = []

print("Inférence sur 100 exemples...")
test_subset = test_bio[:100]

for item in tqdm(test_subset):
    text = " ".join(item["tokens"])
    true_bio = item["labels"]

    entities_pred = extract_entities_mistral(text)

    spans = []
    for ent in entities_pred:
        start = text.lower().find(ent["text"].lower())
        if start != -1:
            end = start + len(ent["text"])
            spans.append({"start": start, "end": end, "label": ent["label"]})

    _, pred_bio = text_to_bio(text, spans)

    # Ajuster les longueurs
    min_len = min(len(true_bio), len(pred_bio))
    true_labels.append(true_bio[:min_len])
    pred_labels.append(pred_bio[:min_len])

# ============================================
# NETTOYAGE DES LABELS
# ============================================

true_labels_clean = [clean_sequence(seq) for seq in true_labels]
pred_labels_clean = [clean_sequence(seq) for seq in pred_labels]

# ============================================
# AFFICHAGE DES RÉSULTATS
# ============================================

print("\n" + "="*50)
print("RÉSULTATS MISTRAL-7B SUR BC5CDR")
print("="*50)

# Accuracy (tous tokens)
acc = accuracy_score(true_labels_clean, pred_labels_clean)
print(f"\n📈 Accuracy (tous tokens): {acc:.4f}")

# Métriques globales (macro average)
print(f"\n📈 Macro Average (Chemical + Disease):")
print(f"   Precision: {precision_score(true_labels_clean, pred_labels_clean, average='macro'):.4f}")
print(f"   Recall: {recall_score(true_labels_clean, pred_labels_clean, average='macro'):.4f}")
print(f"   F1-score: {f1_score(true_labels_clean, pred_labels_clean, average='macro'):.4f}")

# Métriques pondérées
print(f"\n📈 Weighted Average:")
print(f"   Precision: {precision_score(true_labels_clean, pred_labels_clean, average='weighted'):.4f}")
print(f"   Recall: {recall_score(true_labels_clean, pred_labels_clean, average='weighted'):.4f}")
print(f"   F1-score: {f1_score(true_labels_clean, pred_labels_clean, average='weighted'):.4f}")

# Rapport détaillé
print("\n📊 Classification Report (Chemical & Disease):")
print(classification_report(true_labels_clean, pred_labels_clean))

# Distribution des entités dans le test set
print("\n📈 Distribution des entités dans le test set:")
true_flat = [l for seq in true_labels_clean for l in seq]
for label in ['Chemical', 'Disease']:
    b_count = true_flat.count(f"B-{label}")
    i_count = true_flat.count(f"I-{label}")
    print(f"  {label}: {b_count} débuts, {b_count + i_count} total")

Inférence sur 100 exemples...


100%|██████████| 100/100 [17:46<00:00, 10.66s/it]



RÉSULTATS MISTRAL-7B SUR BC5CDR

📈 Accuracy (tous tokens): 0.9513

📈 Macro Average (Chemical + Disease):
   Precision: 0.8089
   Recall: 0.3629
   F1-score: 0.4909

📈 Weighted Average:
   Precision: 0.8096
   Recall: 0.3584
   F1-score: 0.4866

📊 Classification Report (Chemical & Disease):
              precision    recall  f1-score   support

    Chemical       0.79      0.47      0.59       331
     Disease       0.82      0.26      0.39       361

   micro avg       0.81      0.36      0.50       692
   macro avg       0.81      0.36      0.49       692
weighted avg       0.81      0.36      0.49       692


📈 Distribution des entités dans le test set:
  Chemical: 294 débuts, 377 total
  Disease: 347 débuts, 601 total


In [31]:
def predict_sentence_mistral(text):
    entities = extract_entities_mistral(text)
    spans = []
    for ent in entities:
        # The .find() method is called on 'text' (a string) in this context.
        # If 'text' is a list, this will raise the AttributeError.
        start = text.find(ent["text"])
        if start != -1:
            end = start + len(ent["text"])
            spans.append({"start": start, "end": end, "label": ent["label"]})
    tokens, labels = text_to_bio(text, spans)
    print("\n=== Prédiction Mistral ===\n")
    print(f"Tokens : {tokens}")
    print(f"Tags   : {labels}")

# Test

test_sentences = [
    "The patient with chronic kidney disease was treated with Lisinopril and showed improvement.",
    "Ibuprofen may worsen gastric ulcers in patients with peptic ulcer disease.",
    "She was diagnosed with type 2 diabetes and prescribed Metformin and insulin therapy.",
    "Long-term use of corticosteroids like Prednisone can lead to osteoporosis.",
    "The combination of Aspirin and Warfarin increases the risk of bleeding complications."
]

# Loop through each sentence and call the function individually
for sentence in test_sentences:
    predict_sentence_mistral(sentence)


=== Prédiction Mistral ===

Tokens : ['The', 'patient', 'with', 'chronic', 'kidney', 'disease', 'was', 'treated', 'with', 'Lisinopril', 'and', 'showed', 'improvement', '.']
Tags   : ['O', 'O', 'O', 'B-Disease', 'I-Disease', 'I-Disease', 'O', 'O', 'O', 'B-Chemical', 'O', 'O', 'O', 'O']

=== Prédiction Mistral ===

Tokens : ['Ibuprofen', 'may', 'worsen', 'gastric', 'ulcers', 'in', 'patients', 'with', 'peptic', 'ulcer', 'disease', '.']
Tags   : ['B-Chemical', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-Disease', 'I-Disease', 'I-Disease', 'O']

=== Prédiction Mistral ===

Tokens : ['She', 'was', 'diagnosed', 'with', 'type', '2', 'diabetes', 'and', 'prescribed', 'Metformin', 'and', 'insulin', 'therapy', '.']
Tags   : ['O', 'O', 'O', 'O', 'O', 'O', 'B-Disease', 'O', 'O', 'B-Chemical', 'O', 'B-Chemical', 'O', 'O']

=== Prédiction Mistral ===

Tokens : ['Long', '-', 'term', 'use', 'of', 'corticosteroids', 'like', 'Prednisone', 'can', 'lead', 'to', 'osteoporosis', '.']
Tags   : ['O', 'O', 'O', 'O', 

### Displaying BIO tags for each sentence

In [30]:
# Loop through each sentence and call the function individually
for sentence in test_sentences:
    print(f"\n--- Original Sentence ---\n{sentence}")
    predict_sentence_mistral(sentence)


--- Original Sentence ---
The patient with chronic kidney disease was treated with Lisinopril and showed improvement.

=== Prédiction Mistral ===

The                  -> O
patient              -> O
with                 -> O
chronic              -> B-Disease
kidney               -> I-Disease
disease              -> I-Disease
was                  -> O
treated              -> O
with                 -> O
Lisinopril           -> B-Chemical
and                  -> O
showed               -> O
improvement          -> O
.                    -> O

--- Original Sentence ---
Ibuprofen may worsen gastric ulcers in patients with peptic ulcer disease.

=== Prédiction Mistral ===

Ibuprofen            -> B-Chemical
may                  -> O
worsen               -> O
gastric              -> O
ulcers               -> O
in                   -> O
patients             -> O
with                 -> O
peptic               -> B-Disease
ulcer                -> I-Disease
disease              -> I-Disease
.    

In [24]:
from seqeval.metrics import f1_score, precision_score, recall_score, accuracy_score, classification_report
import json
import pandas as pd
import os

MODEL_NAME = "Mistral-7B"

# =========================
# ✅ GLOBAL METRICS
# =========================
precision = precision_score(true_labels_clean, pred_labels_clean)
recall = recall_score(true_labels_clean, pred_labels_clean)
f1 = f1_score(true_labels_clean, pred_labels_clean)
accuracy = accuracy_score(true_labels_clean, pred_labels_clean)

# =========================
# ✅ PER-ENTITY METRICS
# =========================
report = classification_report(true_labels_clean, pred_labels_clean, output_dict=True)

chemical = report.get("Chemical", {})
disease = report.get("Disease", {})

results = {
    "model": MODEL_NAME,

    # └ Global
    "precision": float(precision),
    "recall": float(recall),
    "f1_score": float(f1),
    "accuracy": float(accuracy),

    # └ Chemical
    "Chemical_precision": float(chemical.get("precision", 0)),
    "Chemical_recall": float(chemical.get("recall", 0)),
    "Chemical_f1": float(chemical.get("f1-score", 0)),
    "Chemical_support": int(chemical.get("support", 0)),

    # └ Disease
    "Disease_precision": float(disease.get("precision", 0)),
    "Disease_recall": float(disease.get("recall", 0)),
    "Disease_f1": float(disease.get("f1-score", 0)),
    "Disease_support": int(disease.get("support", 0))
}

# =========================
# ✅ AFFICHAGE
# =========================
print("\n" + "="*50)
print(f"Ÿ R%C9SULTATS {MODEL_NAME}")
print("="*50)
print(f"\n└ Global Metrics:")
print(f"   Precision: {precision:.4f}")
print(f"   Recall: {recall:.4f}")
print(f"   F1-score: {f1:.4f}")
print(f"   Accuracy: {accuracy:.4f}")

print(f"\n└ Chemical:")
print(f"   Precision: {chemical.get('precision', 0):.4f}")
print(f"   Recall: {chemical.get('recall', 0):.4f}")
print(f"   F1-score: {chemical.get('f1-score', 0):.4f}")
print(f"   Support: {chemical.get('support', 0)}")

print(f"\n└ Disease:")
print(f"   Precision: {disease.get('precision', 0):.4f}")
print(f"   Recall: {disease.get('recall', 0):.4f}")
print(f"   F1-score: {disease.get('f1-score', 0):.4f}")
print(f"   Support: {disease.get('support', 0)}")

# =========================
# ✅ SAUVEGARDE
# =========================
os.makedirs("results", exist_ok=True)

json_path = f"results/{MODEL_NAME}.json"
csv_path = f"results/{MODEL_NAME}.csv"

with open(json_path, "w") as f:
    json.dump(results, f, indent=4)

pd.DataFrame([results]).to_csv(csv_path, index=False)

print("\n" + "="*50)
print(f"✅ R%C9sultats {MODEL_NAME} sauvegard%E9s !")
print(f"Ÿ JSON: {json_path}")
print(f"Ÿ CSV: {csv_path}")
print("="*50)



Ÿ R%C9SULTATS Mistral-7B

└ Global Metrics:
   Precision: 0.8052
   Recall: 0.3584
   F1-score: 0.4960
   Accuracy: 0.9513

└ Chemical:
   Precision: 0.7949
   Recall: 0.4683
   F1-score: 0.5894
   Support: 331

└ Disease:
   Precision: 0.8230
   Recall: 0.2576
   F1-score: 0.3924
   Support: 361

✅ R%C9sultats Mistral-7B sauvegard%E9s !
Ÿ JSON: results/Mistral-7B.json
Ÿ CSV: results/Mistral-7B.csv
